<a href="https://colab.research.google.com/github/ameesha543/Statistical-Learning-e23095/blob/main/Assignment_7d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Prior Belief Boundaries

The initial prior distribution is  
$$\Theta \sim \text{Beta}(8, 1.5).$$

The general Beta probability density function is  
$$f_{\Theta}^{(0)}(\theta) = \frac{1}{B(a, b)} \theta^{a-1} (1 - \theta)^{b-1}, \quad 0 < \theta < 1,$$
where  
$$B(a, b) = \frac{\Gamma(a) \Gamma(b)}{\Gamma(a + b)}.$$

Therefore, for $a = 8$ and $b = 1.5$,  
$$f_{\Theta}^{(0)}(\theta) = \frac{1}{B(8, 1.5)} \theta^7 (1 - \theta)^{0.5}$$
for $0 < \theta < 1$.

# Expected prior stiffness

For a Beta distribution,  
$$E[\Theta] = \frac{a}{a + b}.$$
Thus,

$$E[\Theta^{(0)}] = \frac{8}{8 + 1.5} = \frac{8}{9.5}.$$

Therefore,

$$E[\Theta^{(0)}] \approx 0.8421$$

or approximately 84.21% stiffness efficiency.

The prior mode is

$$\theta_{\text{mode}} = \frac{a - 1}{a + b - 2} = \frac{7}{7.5} \approx 0.9333.$$

This means the most likely prior value is approximately 93.33%, indicating that engineers initially believe the structure is likely to be close to its undamaged condition. However, the Beta distribution still assigns some probability to lower stiffness values, allowing the Bayesian model to adapt when evidence of damage is observed.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Beta prior parameters
a = 8.0
b = 1.5

# Avoid theta = 0 because the measurement model contains log(theta)
theta_grid = np.linspace(0.01, 1.0, 5000)

# Evaluate the prior density
prior_density = beta.pdf(theta_grid, a, b)

# Numerically normalize over the computational grid
prior_density = prior_density / np.trapezoid(
    prior_density, theta_grid
)

# Plot the prior
fig_prior = go.Figure()

fig_prior.add_trace(
    go.Scatter(
        x=theta_grid,
        y=prior_density,
        mode="lines",
        name="Beta(8, 1.5) prior"
    )
)

fig_prior.add_vline(
    x=a / (a + b),
    line_dash="dash",
    annotation_text="Prior mean = 0.8421"
)

fig_prior.add_vline(
    x=(a - 1) / (a + b - 2),
    line_dash="dot",
    annotation_text="Prior mode = 0.9333"
)

fig_prior.update_layout(
    title="Initial Prior Distribution of Structural Stiffness",
    xaxis_title="Remaining stiffness factor, θ",
    yaxis_title="Probability density",
    template="plotly_white"
)

fig_prior.show()

## 2. Structural Likelihood Formulation

The measurement model is  
$$y_k = \theta K_{\text{nominal}} e^{\epsilon_k}.$$

Taking natural logarithms gives  
$$\ln y_k = \ln(\theta K_{\text{nominal}}) + \epsilon_k.$$

Since  
$$\epsilon_k \sim \mathcal{N}(0, \sigma^2),$$

it follows that  
$$\ln y_k \mid \theta \sim \mathcal{N}\left(\ln(\theta K_{\text{nominal}}), \sigma^2\right).$$

Therefore,  
$$y_k \mid \theta \sim \text{Lognormal}\left(\ln(\theta K_{\text{nominal}}), \sigma^2\right).$$

---

### Single-measurement likelihood

The likelihood contribution of one measurement is  
$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left[-\frac{(\ln y_k - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2}\right]$$

for $y_k > 0$.
Equivalently,

$$L(y_k | \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp \left[ -\frac{\ln \left( \frac{y_k}{\theta K_{\text{nominal}}} \right)^2}{2\sigma^2} \right]$$

---

### Joint likelihood for the measurement history

Assuming the measurements are conditionally independent when $\theta$ is known,

$$L \left( y^{(k)} | \theta \right) = \prod_{j=1}^k L(y_j | \theta).$$

Thus,

$$L \left( y^{(k)} | \theta \right) = \prod_{j=1}^k \frac{1}{y_j \sigma \sqrt{2\pi}} \exp \left[ -\frac{\ln \left( \frac{y_j}{\theta K_{\text{nominal}}} \right)^2}{2\sigma^2} \right]$$

or, more compactly,

$$L \left( y^{(k)} | \theta \right) = \frac{\exp \left[ -\frac{1}{2\sigma^2} \sum_{j=1}^k \ln \left( \frac{y_j}{\theta K_{\text{nominal}}} \right) \right]}{(2\pi)^{k/2} \sigma^k \prod_{j=1}^k y_j}$$
# 3. Mathematical Formulation of the Non-Conjugate Grid Update

Bayes' theorem gives the posterior distribution after observing the first $k$ measurements:

$$f_{\Theta|Y^{(k)}} \left( \theta \mid y^{(k)} \right) = \frac{L \left( y^{(k)} \mid \theta \right) f_{\Theta}^{(0)}(\theta)}{\int_0^1 L \left( y^{(k)} \mid u \right) f_{\Theta}^{(0)}(u) \, du}.$$

The Beta prior is conjugate to Bernoulli and binomial likelihoods. However, it is not conjugate to the present log-normal likelihood.

The Beta prior has the form

$$\theta^{a-1}(1 - \theta)^{b-1},$$

whereas the likelihood contains

$$\exp \left[ -\frac{(\ln y_k - \ln \theta - \ln K_{\text{nominal}})^2}{2\sigma^2} \right].$$

The terms involving $\ln \theta$ and $(\ln \theta)^2$ cannot be combined with the Beta prior to produce another standard Beta distribution. Therefore, no standard closed-form posterior distribution exists.

# Recursive Bayesian update

Let

$$f_{k-1}(\theta) = f_{\Theta|Y^{(k-1)}} \left( \theta \mid y^{(k-1)} \right)$$

be the posterior after the first $k - 1$ measurements.
When the new measurement $y_k$ is observed, the unnormalized posterior is

$$\tilde{f}_k(\theta) = L(y_k \mid \theta) f_{k-1}(\theta).$$

The normalization constant is

$$Z_k = \int_0^1 L(y_k \mid u) f_{k-1}(u) \, du.$$

Therefore, the normalized recursive posterior is

$$f_k(\theta) = \frac{L(y_k \mid \theta) f_{k-1}(\theta)}{\int_0^1 L(y_k \mid u) f_{k-1}(u) \, du}.$$

with the initial condition

$$f_0(\theta) = f_{\Theta}^{(0)}(\theta).$$

This means that the posterior obtained after one measurement becomes the prior for the next measurement.
# 4. Running Point Estimates

## 4.1 Running posterior mean

The Bayesian posterior-mean estimate after $k$ measurements is

$$\hat{\theta}_{\text{Bayes}}^{(k)} = E[\Theta \mid \mathbf{y}^{(k)}] = \int_{0}^{1} \theta f_k(\theta) \, d\theta$$

where the posterior density satisfies

$$\int_{0}^{1} f_k(\theta) \, d\theta = 1.$$

In terms of the previous posterior and the new likelihood,

$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\int_{0}^{1} \theta L(y_k \mid \theta) f_{k-1}(\theta) \, d\theta}{\int_{0}^{1} L(y_k \mid \theta) f_{k-1}(\theta) \, d\theta}.$$

The posterior mean uses the entire posterior distribution and minimizes the posterior expected squared-error loss.
### 4.2 Running maximum a posteriori estimate

The MAP estimate is the value of $\theta$ at which the posterior density reaches its maximum:

$$\hat{\theta}_{MAP}^{(k)} = \arg \max_{0 < \theta \leq 1} f_k(\theta).$$

Since the normalization constant does not depend on $\theta$,

$$\hat{\theta}_{MAP}^{(k)} = \arg \max_{0 < \theta \leq 1} [L(y_k | \theta) f_{k-1}(\theta)].$$

Unlike the posterior mean, the MAP estimator does not require an additional integration once the normalized posterior grid has been calculated.
# 5. Algorithmic Grid Approximation and Normalization

Choose a grid of $m$ possible stiffness values:

$$\theta_1, \theta_2, \dots, \theta_m.$$

For this problem, a suitable grid is

$$\theta_i \in [0.01, 1.0].$$

The lower limit is set to $0.01$, rather than exactly zero, because the likelihood contains $\ln \theta$, which is undefined at $\theta = 0$.

The upper endpoint $\theta = 1$ can be included because the likelihood is finite there. For the Beta(8, 1.5) prior, the density approaches zero at $\theta = 1$.

## Step 1: Construct the grid

$$\theta_i = 0.01 + (i - 1)\Delta\theta,$$

where

$$\Delta\theta = \frac{1 - 0.01}{m - 1}.$$

A grid containing $m = 5000$ points gives sufficient numerical resolution.
# Step 2: Evaluate the initial prior

$$p_i^{(0)} = f_{\Theta}^{(0)}(\theta_i).$$

The numerical prior is normalized using

$$p_i^{(0)} \leftarrow \frac{p_i^{(0)}}{\text{Trapz}(p^{(0)}, \theta)}.$$

# Step 3: Evaluate the new likelihood

For the new reading $y_k$,

$$\ell_i^{(k)} = L(y_k \mid \theta_i).$$

# Step 4: Calculate the unnormalized posterior

$$\tilde{p}_i^{(k)} = p_i^{(k-1)} \ell_i^{(k)}.$$

# Step 5: Calculate the normalization constant

Using the trapezoidal rule,

$$Z_k \approx \text{Trapz}(\tilde{p}^{(k)}, \theta).$$
# Step 6: Normalize the posterior

$$p_i^{(k)} = \frac{\tilde{p}_i^{(k)}}{Z_k}$$

This ensures that

$$\text{Trapz}(p^{(k)}, \theta) \approx 1.$$

# Step 7: Calculate the point estimates

The posterior mean is approximated as

$$\hat{\theta}_i^{(k)} \approx \text{Trapz}(\theta p^{(k)}, \theta).$$

The MAP estimate is

$$\hat{\theta}_{\text{MAP}}^{(k)} \approx \theta_{i^*},$$

where

$$i^* = \arg \max_i p_i^{(k)}.$$

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# ============================================================
# 1. Problem parameters
# ============================================================

theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n = 15

# Prior parameters
a = 8.0
b = 1.5

# Random seed for reproducible results
rng = np.random.default_rng(42)

# ============================================================
# 2. Generate the sensor measurements
# ============================================================

epsilon = rng.normal(
    loc=0.0,
    scale=sigma,
    size=n
)

measurements = (
    theta_true
    * K_nominal
    * np.exp(epsilon)
)

print("Simulated sensor readings:")
for k, y_k in enumerate(measurements, start=1):
    print(f"k = {k:2d}, y_k = {y_k:.4f} kN/mm")


# ============================================================
# 3. Create the bounded theta grid
# ============================================================

# theta = 0 is excluded because log(theta) is undefined
theta_grid = np.linspace(0.01, 1.0, 5000)


# ============================================================
# 4. Define the single-observation likelihood
# ============================================================

def lognormal_likelihood(y_k, theta, K, noise_sigma):
    """
    Calculate L(y_k | theta) for the log-normal
    structural measurement model.
    """

    if y_k <= 0:
        raise ValueError("The sensor measurement must be positive.")

    log_residual = np.log(
        y_k / (theta * K)
    )

    coefficient = (
        1.0
        / (y_k * noise_sigma * np.sqrt(2.0 * np.pi))
    )

    likelihood = coefficient * np.exp(
        -(log_residual ** 2)
        / (2.0 * noise_sigma ** 2)
    )

    return likelihood


# ============================================================
# 5. Function for posterior credible intervals
# ============================================================

def posterior_quantile(theta, density, probability):
    """
    Estimate a posterior quantile from a grid density
    using cumulative trapezoidal integration.
    """

    interval_widths = np.diff(theta)

    trapezoid_areas = (
        0.5
        * (density[:-1] + density[1:])
        * interval_widths
    )

    cumulative_probability = np.concatenate(
        ([0.0], np.cumsum(trapezoid_areas))
    )

    cumulative_probability /= cumulative_probability[-1]

    return np.interp(
        probability,
        cumulative_probability,
        theta
    )


# ============================================================
# 6. Construct and normalize the initial prior
# ============================================================

posterior = beta.pdf(theta_grid, a, b)

prior_normalization = np.trapezoid(
    posterior,
    theta_grid
)

if prior_normalization <= 0:
    raise RuntimeError("The prior could not be normalized.")

posterior = posterior / prior_normalization


# Store the posterior at every step
posterior_history = [posterior.copy()]

# Step 0 estimates
posterior_means = [
    np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )
]

posterior_maps = [
    theta_grid[np.argmax(posterior)]
]

lower_limits = [
    posterior_quantile(
        theta_grid,
        posterior,
        0.025
    )
]

upper_limits = [
    posterior_quantile(
        theta_grid,
        posterior,
        0.975
    )
]


# ============================================================
# 7. Sequential Bayesian updating
# ============================================================

for y_k in measurements:

    # Evaluate the likelihood over the full theta grid
    likelihood = lognormal_likelihood(
        y_k,
        theta_grid,
        K_nominal,
        sigma
    )

    # Unnormalized posterior
    unnormalized_posterior = (
        posterior * likelihood
    )

    # Normalization using the trapezoidal rule
    normalizing_constant = np.trapezoid(
        unnormalized_posterior,
        theta_grid
    )

    if (
        not np.isfinite(normalizing_constant)
        or normalizing_constant <= 0
    ):
        raise RuntimeError(
            "Posterior normalization failed."
        )

    # Normalized posterior
    posterior = (
        unnormalized_posterior
        / normalizing_constant
    )

    posterior_history.append(
        posterior.copy()
    )

    # Posterior mean
    posterior_mean = np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )

    # MAP estimate
    posterior_map = theta_grid[
        np.argmax(posterior)
    ]

    # 95% credible interval
    lower_95 = posterior_quantile(
        theta_grid,
        posterior,
        0.025
    )

    upper_95 = posterior_quantile(
        theta_grid,
        posterior,
        0.975
    )

    posterior_means.append(posterior_mean)
    posterior_maps.append(posterior_map)
    lower_limits.append(lower_95)
    upper_limits.append(upper_95)


# Convert lists into arrays
posterior_means = np.asarray(posterior_means)
posterior_maps = np.asarray(posterior_maps)
lower_limits = np.asarray(lower_limits)
upper_limits = np.asarray(upper_limits)

steps = np.arange(n + 1)


# ============================================================
# 8. Display numerical results
# ============================================================

print("\nSequential estimation results")
print(
    "Step   Posterior mean   MAP estimate"
    "   95% credible interval"
)

for k in range(n + 1):
    print(
        f"{k:2d}"
        f"       {posterior_means[k]:.4f}"
        f"          {posterior_maps[k]:.4f}"
        f"       [{lower_limits[k]:.4f}, "
        f"{upper_limits[k]:.4f}]"
    )


# ============================================================
# 9. Plot posterior densities at selected milestones
# ============================================================

milestones = [0, 1, 2, 5, 10, 15]

fig_density = go.Figure()

for k in milestones:
    fig_density.add_trace(
        go.Scatter(
            x=theta_grid,
            y=posterior_history[k],
            mode="lines",
            name=f"k = {k}"
        )
    )

fig_density.add_vline(
    x=theta_true,
    line_dash="dash",
    annotation_text="True θ = 0.68",
    annotation_position="top"
)

fig_density.update_layout(
    title=(
        "Sequential Evolution of the "
        "Posterior Stiffness Distribution"
    ),
    xaxis_title="Remaining stiffness factor, θ",
    yaxis_title="Posterior probability density",
    template="plotly_white",
    legend_title="Inspection step"
)

fig_density.show()


# ============================================================
# 10. Plot posterior mean and MAP estimates
# ============================================================

fig_estimates = go.Figure()

fig_estimates.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior mean"
    )
)

fig_estimates.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_maps,
        mode="lines+markers",
        name="MAP estimate"
    )
)

fig_estimates.add_trace(
    go.Scatter(
        x=steps,
        y=np.full(n + 1, theta_true),
        mode="lines",
        line=dict(dash="dash"),
        name="True stiffness, θ = 0.68"
    )
)

fig_estimates.update_layout(
    title="Convergence of the Bayesian Stiffness Estimates",
    xaxis_title="Number of sensor readings, k",
    yaxis_title="Estimated stiffness factor",
    xaxis=dict(dtick=1),
    template="plotly_white"
)

fig_estimates.show()

Simulated sensor readings:
k =  1, y_k = 35.5901 kN/mm
k =  2, y_k = 29.0891 kN/mm
k =  3, y_k = 38.0510 kN/mm
k =  4, y_k = 39.1518 kN/mm
k =  5, y_k = 25.3735 kN/mm
k =  6, y_k = 27.9672 kN/mm
k =  7, y_k = 34.6583 kN/mm
k =  8, y_k = 32.4248 kN/mm
k =  9, y_k = 33.9144 kN/mm
k = 10, y_k = 29.9163 kN/mm
k = 11, y_k = 38.7942 kN/mm
k = 12, y_k = 38.2074 kN/mm
k = 13, y_k = 34.3384 kN/mm
k = 14, y_k = 40.2636 kN/mm
k = 15, y_k = 36.4699 kN/mm

Sequential estimation results
Step   Posterior mean   MAP estimate   95% credible interval
 0       0.8421          0.9333       [0.5668, 0.9870]
 1       0.7947          0.7972       [0.6107, 0.9629]
 2       0.6977          0.6877       [0.5661, 0.8476]
 3       0.7177          0.7107       [0.6051, 0.8436]
 4       0.7332          0.7277       [0.6325, 0.8443]
 5       0.6823          0.6780       [0.5974, 0.7754]
 6       0.6604          0.6568       [0.5849, 0.7426]
 7       0.6649          0.6617       [0.5943, 0.7414]
 8       0.6629      